In [ ]:
import numpy as np
from scipy.ndimage import label
from skimage.measure import regionprops
import matplotlib.pyplot as plt
from PIL import Image
import os

# Path to grayscale rock image; pixels below 128 are treated as pore space
img_path = "path/to/your_rock_image.png"

# --------------------------------------------------------
# Load binary image (True = pores)
# --------------------------------------------------------
img = Image.open(img_path).convert("L")
arr = np.array(img)
binary = arr < 128
H, W = binary.shape
img_area = H * W

# --------------------------------------------------------
# Extract pores, filter small ones, store centroids + shapes
# --------------------------------------------------------
labeled, n = label(binary)
regions = regionprops(labeled)

pore_objects = []
area_thresh = 0.0005  # 0.05%

for reg in regions:
    if reg.area / img_area < area_thresh:
        continue  # ignore tiny pore

    # bounding box
    minr, minc, maxr, maxc = reg.bbox
    pore_mask = (labeled[minr:maxr, minc:maxc] == reg.label)

    # centroid (absolute coords)
    cy, cx = reg.centroid

    # relative coordinates of pore pixels to centroid
    ys, xs = np.where(pore_mask)
    rel_y = ys - (cy - minr)
    rel_x = xs - (cx - minc)

    pore_objects.append(dict(
        rel_y=rel_y,
        rel_x=rel_x,
        centroid=np.array([cy, cx])
    ))

print(f"Kept {len(pore_objects)} pores after filtering.")


# --------------------------------------------------------
# Percolation check
# --------------------------------------------------------
def check_percolation(binary_img):
    lab, n = label(binary_img)
    if n == 0:
        return False

    left = set(np.unique(lab[:, 0])) - {0}
    right = set(np.unique(lab[:, -1])) - {0}
    top = set(np.unique(lab[0, :])) - {0}
    bottom = set(np.unique(lab[-1, :])) - {0}

    return (len(left & right) > 0) or (len(top & bottom) > 0)


# --------------------------------------------------------
# Reconstruct image at different scale factors
# --------------------------------------------------------
scale_list = np.linspace(1.0, 0.6, 10)   # e.g. 1.0, 0.9, ..., 0.4

results = []
percolated_at = None
percolated_img = None


for scale in scale_list:
    new_H = max(5, int(H * scale))
    new_W = max(5, int(W * scale))

    new_img = np.zeros((new_H, new_W), dtype=bool)

    for obj in pore_objects:
        cy, cx = obj["centroid"]
        # scaled centroid
        new_cy = cy * scale
        new_cx = cx * scale

        # reconstruct pore footprint pixels
        ys = (new_cy + obj["rel_y"]).astype(int)
        xs = (new_cx + obj["rel_x"]).astype(int)

        valid = (ys >= 0) & (ys < new_H) & (xs >= 0) & (xs < new_W)
        new_img[ys[valid], xs[valid]] = True

    # check percolation
    percolates = check_percolation(new_img)
    results.append((scale, new_img, percolates))

    print(f"Scale {scale:.2f}  => Percolates: {percolates}")

    if percolates and percolated_at is None:
        percolated_at = scale
        percolated_img = new_img.copy()


# --------------------------------------------------------
# Visualization
# --------------------------------------------------------
cols = len(results)
plt.figure(figsize=(4*cols, 6))

for i, (scale, img_step, percol) in enumerate(results):
    ax = plt.subplot(1, cols, i+1)
    ax.imshow(img_step, cmap='gray')
    ax.set_title(f"Scale {scale:.2f}\nPercolates: {percol}")
    ax.axis("off")

plt.tight_layout()
plt.savefig("scaled_overlap_pores.png", dpi=150)
plt.show()

print("Finished. Saved: scaled_overlap_pores.png")
if percolated_at:
    print("Percolation achieved at scale =", percolated_at)
else:
    print("Percolation not achieved.")

# --------------------------------------------------------
# Save the percolation img
# --------------------------------------------------------
if percolated_img is not None:
    save_dir = os.path.dirname(img_path)
    save_path = os.path.join(save_dir, "perco.png")

    # Convert boolean → uint8 image
    out_img = (percolated_img.astype(np.uint8) * 255)
    Image.fromarray(out_img).save(save_path)

    print(f"First percolation image saved to: {save_path}")
else:
    print("No percolating image found — nothing saved.")

